# Web scraping ético de precios en MercadoLibre Colombia

**Artículo piloto:** Apple iPhone 17 Pro Max (256 GB) - Color plata
**Método elegido:** Selenium (renderizado del sitio) + BeautifulSoup (análisis del HTML)
**Ejecución:** Google Colab, Jupyter local o **GitHub Actions** (Python 3, Chrome headless)

---

## Objetivo del cuaderno

Capturar de forma automatizada y respetuosa los datos de **un solo artículo** de MercadoLibre para, en
prácticas posteriores, analizar la variación mensual de sus precios.

**Datos a extraer**

| Campo | Descripción |
|---|---|
| Nombre del artículo | Título de la publicación |
| Precio sin descuento | Precio de lista (tachado) |
| Precio con descuento | Precio vigente de venta |
| Descuento total | Diferencia en COP y porcentaje |
| URL del artículo | Dirección canónica de la publicación |

## Mapa del cuaderno

1. Instalación del entorno
2. Configuración y parámetros
3. Verificación ética (`robots.txt`)
4. Prueba de viabilidad: `requests` + BeautifulSoup
5. Extracción con Selenium
6. Análisis del HTML con BeautifulSoup
7. Prueba del analizador con HTML de muestra
8. Validación y métricas de éxito
9. Almacenamiento histórico (CSV)
10. Conclusiones y trabajo futuro

> **Cómo usarlo:** abre el cuaderno en Google Colab y ejecuta las celdas en orden
> (*Entorno de ejecución → Ejecutar todo*). Una ejecución completa toma 1–2 minutos.

> **Modo de ejecución (variable de entorno `SCRAPING_MODO`):**
> `en_vivo` (por defecto) consulta el sitio una vez; `offline` **no** hace ninguna petición y analiza el HTML guardado en
> `data/pagina_articulo.html`. El pipeline de GitHub Actions usa `offline` en cada *push*/*pull request* y `en_vivo`
> solo en la ejecución mensual programada o disparada manualmente.

## 1. Instalación del entorno

Colab no trae Google Chrome. En esta celda:

- Se descarga e instala **Google Chrome estable** (necesario para que Selenium renderice la página).
- Se instalan las librerías de Python: `selenium` (control del navegador), `beautifulsoup4` (análisis de HTML),
  `requests` (prueba de viabilidad y `robots.txt`) y `pandas` (tabla de resultados).

Selenium 4 incluye *Selenium Manager*, que descarga automáticamente el `chromedriver` compatible con la versión de
Chrome instalada, por lo que no hay que configurarlo a mano.

En Colab se instala todo aquí. En local o en GitHub Actions las dependencias vienen de `requirements.txt` y Chrome ya está instalado, por lo que la celda no hace nada.

In [ ]:
import sys, subprocess

EN_COLAB = "google.colab" in sys.modules

if EN_COLAB:
    subprocess.run("wget -q -O /tmp/google-chrome.deb https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb"
                   " && apt-get install -y -qq /tmp/google-chrome.deb",
                   shell=True, check=True, stdout=subprocess.DEVNULL)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "selenium", "beautifulsoup4", "requests", "pandas"],
                   check=True)
    print("Entorno de Colab preparado (Chrome + librerías).")
else:
    print("Entorno no-Colab: se usan las dependencias de requirements.txt y el Chrome del sistema.")

In [ ]:
# Verificación rápida de la instalación
import selenium, bs4
try:
    chrome = subprocess.run(["google-chrome", "--version"], capture_output=True, text=True).stdout.strip()
except FileNotFoundError:
    chrome = "no instalado (solo se necesita en modo en_vivo)"
print("chrome:", chrome)
print("selenium:", selenium.__version__, "| beautifulsoup4:", bs4.__version__)

## 2. Configuración y parámetros

Se declaran en un único lugar los parámetros de la práctica:

- **`URL_ARTICULO`**: la URL original incluye parámetros de seguimiento (`navigation_id`, `polycard_client`, etc.)
  que solo sirven para analítica de MercadoLibre. Se usa la **URL limpia** del producto (`/p/MCO55308605`), que es
  estable en el tiempo y evita enviar datos de navegación innecesarios.
- **`USER_AGENT_SUFIJO`**: texto que identifica honestamente el propósito de las peticiones.
- **`PAUSA_SEGUNDOS`**: espera cortés antes de cada petición.
- **`ARCHIVO_CSV`**: archivo donde se acumulará el histórico de precios.
- **`MODO`**: `en_vivo` consulta el sitio; `offline` solo analiza el HTML guardado (sin peticiones).
- **`ARCHIVO_HTML`**: captura del HTML renderizado, versionada en `data/`.

In [ ]:
import os
import re
import time
import json
from datetime import datetime
from urllib.parse import urlparse

import requests
import pandas as pd
from bs4 import BeautifulSoup

# --- Parámetros de la práctica ---
URL_ARTICULO = "https://www.mercadolibre.com.co/apple-iphone-17-pro-max-256-gb-color-plata/p/MCO55308605"
ID_PRODUCTO = "MCO55308605"
NOMBRE_ESPERADO = "Apple iPhone 17 Pro Max (256 GB) - Color plata"

USER_AGENT_SUFIJO = "PracticaAcademicaWebScraping/1.0 (uso educativo; una sola URL)"
PAUSA_SEGUNDOS = 3
TIEMPO_MAX_ESPERA = 30          # segundos máximos esperando que cargue el precio
ARCHIVO_CSV = "precios_iphone_17_pro_max.csv"

print("URL del artículo:", URL_ARTICULO)
print("Fecha/hora local :", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
# --- Modo de ejecución (lo define el pipeline de GitHub Actions) ---
MODO = os.getenv("SCRAPING_MODO", "en_vivo").lower()      # "en_vivo" | "offline"
MODO_OFFLINE = MODO == "offline"
EN_CI = os.getenv("CI", "").lower() == "true"             # GitHub Actions define CI=true
ARCHIVO_HTML = "data/pagina_articulo.html"               # captura guardada del navegador (plan alternativo)
print("Modo de ejecución:", MODO, "| en CI:", EN_CI)

## 3. Verificación ética: `robots.txt`

Antes de pedir cualquier página se consulta el archivo `robots.txt` del sitio, que indica qué rutas pueden recorrer
los robots. La celda:

1. Descarga `https://www.mercadolibre.com.co/robots.txt` identificándose con un *User-Agent* claro.
2. Usa `urllib.robotparser` para comprobar si la ruta del artículo está permitida.
3. **Detiene la ejecución** si la ruta está prohibida. Si el archivo no se puede leer, lo advierte y continúa con
   una única petición de bajo impacto (se documenta en el informe).

Buenas prácticas aplicadas en todo el cuaderno: una sola URL, una sola carga de página por ejecución, pausa previa,
sin evadir captchas ni inicios de sesión, y uso exclusivamente académico de los datos públicos.

In [ ]:
from urllib import robotparser

ROBOTS_URL = "https://www.mercadolibre.com.co/robots.txt"
ruta = urlparse(URL_ARTICULO).path
ua_robots = f"Mozilla/5.0 {USER_AGENT_SUFIJO}"

estado_robots = "no verificado"
if MODO_OFFLINE:
    estado_robots = "omitido (modo offline: no se hacen peticiones)"
else:
    try:
        resp = requests.get(ROBOTS_URL, headers={"User-Agent": ua_robots}, timeout=15)
        if resp.status_code == 200:
            rp = robotparser.RobotFileParser()
            rp.parse(resp.text.splitlines())
            permitido = rp.can_fetch("*", URL_ARTICULO)
            estado_robots = "PERMITIDO" if permitido else "PROHIBIDO"
            espera = rp.crawl_delay("*")
            print(f"robots.txt leído (HTTP 200). Ruta {ruta} -> {estado_robots}")
            if espera:
                print(f"Crawl-delay solicitado por el sitio: {espera} s")
                PAUSA_SEGUNDOS = max(PAUSA_SEGUNDOS, espera)
        else:
            print(f"Aviso: robots.txt respondió HTTP {resp.status_code}; no se pudo verificar la regla.")
    except requests.RequestException as e:
        print("Aviso: no se pudo descargar robots.txt:", e)

if estado_robots == "PROHIBIDO":
    raise RuntimeError("robots.txt prohíbe esta ruta: se detiene la práctica para respetar al sitio.")
print("Estado ético:", estado_robots, "| pausa configurada:", PAUSA_SEGUNDOS, "s")

## 4. Prueba de viabilidad: `requests` + BeautifulSoup

Antes de elegir el método se evalúa la opción más liviana: pedir el HTML con `requests` y analizarlo con
BeautifulSoup. Esta opción solo funciona si el servidor entrega **el precio ya escrito en el HTML**.

La celda hace **una única petición** y revisa si en el HTML recibido aparecen el título y los precios. Interpretación:

- Si **no** aparecen (o el sitio redirige a un inicio de sesión), el contenido se arma con JavaScript en el
  navegador y hace falta un navegador real → **Selenium**.
- Si aparecen, BeautifulSoup solo sería suficiente.

> **Scrapy** no se ejecuta aquí: es un *framework* orientado a recorrer miles de páginas (arañas, colas, *pipelines*).
> Para extraer un solo artículo sería excesivo y, además, tampoco ejecuta JavaScript sin extensiones adicionales.

In [ ]:
diagnostico = {}
if MODO_OFFLINE:
    print("Modo offline: se omite la prueba de viabilidad con requests (no se hace ninguna petición).")
else:
    time.sleep(PAUSA_SEGUNDOS)  # pausa cortés
    try:
        r = requests.get(
            URL_ARTICULO,
            headers={"User-Agent": f"Mozilla/5.0 {USER_AGENT_SUFIJO}", "Accept-Language": "es-CO,es;q=0.9"},
            timeout=20,
        )
        s = BeautifulSoup(r.text, "html.parser")
        diagnostico = {
            "http_status": r.status_code,
            "url_final": r.url,
            "bytes_html": len(r.text),
            "redirige_a_login": "account-verification" in r.url or "login" in urlparse(r.url).path,
            "tiene_titulo_h1": s.select_one("h1.ui-pdp-title") is not None,
            "tiene_precio": s.select_one(".andes-money-amount") is not None,
        }
    except requests.RequestException as e:
        diagnostico = {"error": str(e)}

    for k, v in diagnostico.items():
        print(f"{k:>18}: {v}")

    if diagnostico.get("tiene_titulo_h1") and diagnostico.get("tiene_precio"):
        print("\nConclusión: el HTML estático ya trae los datos; BeautifulSoup + requests sería suficiente.")
    else:
        print("\nConclusión: el HTML estático NO trae los datos del producto -> se necesita renderizar la página (Selenium).")

## 5. Extracción con Selenium

Selenium controla un **Chrome sin interfaz gráfica (*headless*)** que ejecuta el JavaScript de la página, de modo que
el título y los precios quedan escritos en el DOM.

Pasos de la celda:

1. Configurar Chrome: modo *headless*, `--no-sandbox` y `--disable-dev-shm-usage` (necesarios en Colab) e idioma `es-CO`.
2. Identificar las peticiones con el *User-Agent* de la práctica.
3. Cargar la URL **una sola vez** y esperar explícitamente (`WebDriverWait`) a que aparezca el título del producto.
4. Si el sitio muestra un **muro de inicio de sesión o verificación** (algo frecuente cuando la petición sale de
   los servidores de Colab), se avisa con un mensaje claro y se pasa al plan alternativo de la sección 5.1:
   **no se intenta saltarlo** (no se evaden captchas, verificaciones ni autenticaciones).
5. Guardar el HTML renderizado en la variable `html_renderizado` y cerrar el navegador siempre (`finally`).
6. En modo `offline` (usado por el pipeline en cada *push*/*pull request*) este paso se omite por completo.

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, WebDriverException

opciones = Options()
opciones.add_argument("--headless=new")
opciones.add_argument("--no-sandbox")
opciones.add_argument("--disable-dev-shm-usage")
opciones.add_argument("--window-size=1366,900")
opciones.add_argument("--lang=es-CO")

html_renderizado = None
url_final = None
bloqueado = False
origen_datos = "Selenium (automatizado)"
t_inicio = time.time()

if MODO_OFFLINE:
    print("Modo offline: no se abre el navegador ni se consulta el sitio. Se usará el HTML guardado (sección 5.1).")
else:
    driver = None
    try:
        driver = webdriver.Chrome(options=opciones)   # Selenium Manager resuelve el chromedriver
        # Identificación honesta: se conserva el UA de Chrome y se agrega el sufijo de la práctica
        ua_base = driver.execute_script("return navigator.userAgent").replace("HeadlessChrome", "Chrome")
        try:
            driver.execute_cdp_cmd("Network.setUserAgentOverride",
                                   {"userAgent": f"{ua_base} {USER_AGENT_SUFIJO}", "acceptLanguage": "es-CO,es"})
        except Exception:
            pass  # si el override no está disponible se continúa con el UA por defecto

        time.sleep(PAUSA_SEGUNDOS)            # pausa cortés antes de la única petición
        driver.get(URL_ARTICULO)

        if "account-verification" in driver.current_url:
            bloqueado = True
            print("MercadoLibre mostró un muro de inicio de sesión/verificación.")
            print("Por ética NO se intenta evadirlo. Continúa con el plan alternativo (sección 5.1).")
        else:
            WebDriverWait(driver, TIEMPO_MAX_ESPERA).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "h1.ui-pdp-title"))
            )
            WebDriverWait(driver, TIEMPO_MAX_ESPERA).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, ".ui-pdp-price .andes-money-amount"))
            )
            html_renderizado = driver.page_source
            url_final = driver.current_url
    except TimeoutException:
        print("Tiempo de espera agotado: la página no mostró título/precio (¿bloqueo o cambio de estructura?).")
        print("Continúa con el plan alternativo (sección 5.1).")
    except WebDriverException as e:
        print("No se pudo usar el navegador automatizado:", str(e).strip().splitlines()[0][:200])
        print("Continúa con el plan alternativo (sección 5.1).")
    finally:
        if driver is not None:
            driver.quit()

duracion_s = round(time.time() - t_inicio, 1)
print(f"HTML renderizado obtenido: {html_renderizado is not None} | tamaño: {len(html_renderizado or '')} caracteres | {duracion_s} s")

### 5.1 Plan alternativo si el sitio bloquea el acceso automatizado

Si la celda anterior no obtuvo el HTML (muro de verificación, tiempo agotado), **no se insiste ni se evaden los
controles del sitio**. En su lugar, la persona abre la ficha en su navegador normal —como cualquier visitante— y guarda
el HTML ya renderizado; el cuaderno lo analiza con BeautifulSoup exactamente igual que si lo hubiera descargado Selenium.

**Cómo guardar el HTML (una vez por medición):**

1. Abre la URL del artículo en tu navegador (Chrome) y espera a que se vea el precio.
2. Presiona **F12** → pestaña **Elements** (Elementos).
3. Clic derecho sobre la primera línea `<html ...>` → **Copy → Copy outerHTML**.
4. Pega el contenido en un editor de texto simple y guárdalo como `pagina_articulo.html`.
5. En Colab, ejecuta la celda siguiente y selecciona ese archivo cuando se solicite (o súbelo antes con el ícono de
   carpeta de la barra lateral).

> Alternativa: ejecutar este mismo cuaderno de forma local (Jupyter en tu computador, con Chrome instalado), donde la
> petición sale de tu conexión habitual. Si el sitio vuelve a mostrar el muro, se usa este plan alternativo.

> En este repositorio la captura vigente está en **`data/pagina_articulo.html`** (HTML renderizado del 27/10/2026). Es el archivo que usa el modo `offline` y el que se carga si el sitio bloquea el acceso automatizado desde los servidores de GitHub.

In [ ]:
if html_renderizado is None:
    if os.path.exists(ARCHIVO_HTML):
        with open(ARCHIVO_HTML, "r", encoding="utf-8", errors="ignore") as f:
            html_renderizado = f.read()
        origen_datos = "HTML guardado manualmente desde el navegador"
        print(f"HTML cargado desde {ARCHIVO_HTML}: {len(html_renderizado)} caracteres.")
    elif EN_COLAB:
        from google.colab import files
        print(f"Selecciona el archivo HTML guardado desde tu navegador (se guardará como {ARCHIVO_HTML}):")
        subidos = files.upload()
        if subidos:
            os.makedirs(os.path.dirname(ARCHIVO_HTML), exist_ok=True)
            nombre = next(iter(subidos))
            with open(ARCHIVO_HTML, "wb") as f:
                f.write(subidos[nombre])
            with open(ARCHIVO_HTML, "r", encoding="utf-8", errors="ignore") as f:
                html_renderizado = f.read()
            origen_datos = "HTML guardado manualmente desde el navegador"
    else:
        print(f"No hay HTML disponible: guarda la página como {ARCHIVO_HTML} y vuelve a ejecutar la celda.")
else:
    print("El HTML ya se obtuvo con Selenium; no se necesita el plan alternativo.")

## 6. Análisis del HTML con BeautifulSoup

Con el HTML ya renderizado, BeautifulSoup extrae cada dato mediante **selectores CSS** identificados al inspeccionar la
página del artículo:

| Dato | Selector CSS | Observación |
|---|---|---|
| Nombre | `h1.ui-pdp-title` | Título de la publicación |
| Precio sin descuento | `.ui-pdp-price .andes-money-amount--previous` | Elemento tachado; el atributo `aria-label` trae el valor limpio (p. ej. «Antes: 6999000 pesos») |
| Precio con descuento | `meta[itemprop=price]` (o `.ui-pdp-price__second-line .andes-money-amount`) | Precio vigente |
| Descuento (sitio) | `.andes-money-amount__discount` | Texto del tipo «36% OFF» |
| URL | `link[rel=canonical]` | Dirección canónica de la página |

Reglas de cálculo:

- `descuento_total_cop = precio_sin_descuento − precio_con_descuento`
- `descuento_pct_calculado = descuento_total_cop / precio_sin_descuento × 100`
- Si la publicación no tuviera precio tachado, se asume sin descuento (precio de lista = precio vigente).

Se define la función `extraer_datos` para poder reutilizarla y probarla de forma independiente.

In [ ]:
def a_entero(texto):
    """Convierte textos como '6.999.000' o 'Antes: 6999000 pesos' en el entero 6999000."""
    digitos = re.sub(r"[^\d]", "", texto or "")
    return int(digitos) if digitos else None


def extraer_datos(html, url_respaldo):
    """Analiza el HTML renderizado de la ficha de producto y devuelve un diccionario con los campos solicitados."""
    sopa = BeautifulSoup(html, "html.parser")
    bloque = sopa.select_one(".ui-pdp-price") or sopa

    # --- Nombre ---
    h1 = sopa.select_one("h1.ui-pdp-title")
    nombre = h1.get_text(strip=True) if h1 else None

    # --- Precio con descuento (vigente) ---
    precio_actual = None
    meta = bloque.select_one("meta[itemprop=price]") or sopa.select_one("meta[itemprop=price]")
    if meta and meta.get("content"):
        precio_actual = a_entero(meta["content"].split(".")[0])
    if precio_actual is None:
        el = bloque.select_one(".ui-pdp-price__second-line .andes-money-amount")
        if el:
            precio_actual = a_entero(el.get("aria-label")) or a_entero(
                el.select_one(".andes-money-amount__fraction").get_text() if el.select_one(".andes-money-amount__fraction") else None)

    # --- Precio sin descuento (tachado) ---
    precio_lista = None
    previo = bloque.select_one(".andes-money-amount--previous")
    if previo:
        precio_lista = a_entero(previo.get("aria-label"))
        if precio_lista is None:
            frac = previo.select_one(".andes-money-amount__fraction")
            precio_lista = a_entero(frac.get_text() if frac else None)
    if precio_lista is None:
        precio_lista = precio_actual  # sin precio tachado -> sin descuento

    # --- Descuento ---
    desc_cop = (precio_lista - precio_actual) if (precio_lista is not None and precio_actual is not None) else None
    desc_pct_calc = round(100 * desc_cop / precio_lista, 2) if desc_cop is not None and precio_lista else None
    etiqueta = bloque.select_one(".andes-money-amount__discount")
    m = re.search(r"(\d+(?:[.,]\d+)?)\s*%", etiqueta.get_text()) if etiqueta else None
    desc_pct_sitio = float(m.group(1).replace(",", ".")) if m else (0.0 if desc_cop == 0 else None)

    # --- URL canónica ---
    canonica = sopa.select_one("link[rel=canonical]")
    url_articulo = canonica["href"] if canonica and canonica.get("href") else url_respaldo

    return {
        "fecha_extraccion": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "nombre": nombre,
        "precio_sin_descuento": precio_lista,
        "precio_con_descuento": precio_actual,
        "descuento_total_cop": desc_cop,
        "descuento_pct_sitio": desc_pct_sitio,
        "descuento_pct_calculado": desc_pct_calc,
        "moneda": "COP",
        "url": url_articulo,
    }

## 7. Prueba del analizador con HTML de muestra

Para comprobar que `extraer_datos` funciona **independientemente de la red**, se prueba con un fragmento de HTML que
reproduce la estructura observada en la página real (clases CSS y atributos), con los valores vistos el 27/10/2026.
Si esta prueba falla, el problema está en el código; si pasa y la extracción real falla, el problema está en el acceso
al sitio o en un cambio de su estructura.

In [ ]:
HTML_MUESTRA = """
<html><head>
<link rel="canonical" href="https://www.mercadolibre.com.co/apple-iphone-17-pro-max-256-gb-color-plata/p/MCO55308605"/>
</head><body>
<h1 class="ui-pdp-title">Apple iPhone 17 Pro Max (256 GB) - Color plata</h1>
<div class="ui-pdp-price spacing-layout mt-16 ui-pdp-price--size-large">
  <div class="ui-pdp-price__main-container">
    <s class="andes-money-amount ui-pdp-price__part ui-pdp-price__original-value andes-money-amount--previous"
       role="img" aria-label="Antes: 6999000 pesos">
      <span class="andes-money-amount__currency"><span class="andes-money-amount__currency-symbol">$</span></span>
      <span class="andes-money-amount__fraction">6.999.000</span>
    </s>
    <div class="ui-pdp-price__second-line">
      <span class="ui-pdp-price__part__container">
        <span class="andes-money-amount ui-pdp-price__part andes-money-amount--cents-superscript" aria-label="4409900 pesos">
          <meta itemprop="price" content="4409900"/>
          <span class="andes-money-amount__fraction">4.409.900</span>
        </span>
      </span>
      <span class="ui-pdp-price__second-line__label">
        <span class="andes-money-amount__discount ui-pdp-family--SEMIBOLD">36% OFF</span>
      </span>
    </div>
  </div>
</div>
</body></html>
"""

prueba = extraer_datos(HTML_MUESTRA, URL_ARTICULO)
assert prueba["nombre"] == NOMBRE_ESPERADO
assert prueba["precio_sin_descuento"] == 6999000
assert prueba["precio_con_descuento"] == 4409900
assert prueba["descuento_total_cop"] == 2589100
assert prueba["descuento_pct_sitio"] == 36.0
print("Prueba del analizador superada:")
print(json.dumps(prueba, indent=2, ensure_ascii=False))

## 8. Datos reales, validación y métricas de éxito

Se aplica `extraer_datos` al HTML renderizado por Selenium y se evalúa la calidad del resultado con métricas
objetivas. La extracción se considera **exitosa** cuando se cumplen todas:

| Métrica | Criterio de éxito |
|---|---|
| **Completitud** | Los 5 campos solicitados están presentes (100 %) |
| **Coherencia de precios** | Precio con descuento ≤ precio sin descuento |
| **Consistencia del descuento** | Diferencia entre el % publicado por el sitio y el % calculado ≤ 1 punto (el sitio trunca decimales) |
| **Identidad del artículo** | El nombre coincide con el esperado y la URL contiene el identificador `MCO55308605` |
| **Eficiencia / cortesía** | 1 única carga de página y duración total < 60 s (solo aplica cuando el HTML lo obtiene Selenium) |

In [ ]:
if html_renderizado is None:
    raise RuntimeError("No hay HTML renderizado: revisa la celda de Selenium (bloqueo, tiempo de espera o red).")

datos = extraer_datos(html_renderizado, URL_ARTICULO)
datos["origen_datos"] = origen_datos
print("Origen del HTML:", origen_datos)

campos_obligatorios = ["nombre", "precio_sin_descuento", "precio_con_descuento", "descuento_total_cop", "url"]
completitud = 100 * sum(datos[c] is not None for c in campos_obligatorios) / len(campos_obligatorios)

coherente = (datos["precio_con_descuento"] is not None and datos["precio_sin_descuento"] is not None
             and datos["precio_con_descuento"] <= datos["precio_sin_descuento"])

dif_pct = None
if datos["descuento_pct_sitio"] is not None and datos["descuento_pct_calculado"] is not None:
    dif_pct = round(abs(datos["descuento_pct_sitio"] - datos["descuento_pct_calculado"]), 2)
consistente = dif_pct is not None and dif_pct <= 1.0

identidad = (datos["nombre"] == NOMBRE_ESPERADO) and (ID_PRODUCTO in (datos["url"] or ""))
automatizado = origen_datos.startswith("Selenium")
eficiente = (duracion_s < 60) if automatizado else True   # con HTML manual no hay carga automatizada que medir

metricas = pd.DataFrame([
    ["Completitud de campos", f"{completitud:.0f} %", completitud == 100],
    ["Coherencia de precios", "con descuento <= sin descuento", coherente],
    ["Consistencia del descuento", f"diferencia {dif_pct} pp" if dif_pct is not None else "sin dato", consistente],
    ["Identidad del artículo", "nombre y URL esperados", identidad],
    ["Eficiencia (1 carga, < 60 s)", f"{duracion_s} s" if automatizado else "no aplica (HTML manual)", eficiente],
], columns=["Métrica", "Resultado", "Cumple"])
display(metricas)

print("EXTRACCIÓN EXITOSA" if metricas["Cumple"].all() else "REVISAR: alguna métrica no se cumplió")

## 9. Almacenamiento histórico (CSV)

El resultado se **agrega** (no sobrescribe) a un archivo CSV: cada ejecución añade una fila con la fecha y hora. Así,
ejecutando este cuaderno periódicamente (por ejemplo, una vez al mes, en línea con el objetivo del proyecto) se
construye la serie histórica de precios que se analizará en prácticas posteriores.

> En Colab el archivo vive en el almacenamiento temporal de la sesión; la celda ofrece su descarga para conservarlo.

> **En GitHub Actions** el CSV solo se acumula cuando Selenium obtiene los datos en vivo; con el modo `offline` o con el HTML de respaldo no se agrega fila, para no registrar como medición nueva una captura antigua. El CSV queda como artefacto de la ejecución.

In [ ]:
fila = pd.DataFrame([datos])

# En CI solo se acumula histórico si el dato lo obtuvo Selenium en vivo: ni el modo offline
# ni el HTML de respaldo (captura antigua) deben contarse como una medición nueva.
GUARDAR_CSV = not MODO_OFFLINE and (automatizado or not EN_CI)

if GUARDAR_CSV:
    existe = os.path.exists(ARCHIVO_CSV)
    fila.to_csv(ARCHIVO_CSV, mode="a", header=not existe, index=False, encoding="utf-8-sig")
    historico = pd.read_csv(ARCHIVO_CSV)
    print(f"Registros acumulados en {ARCHIVO_CSV}: {len(historico)}")
else:
    print("CSV histórico NO actualizado en esta ejecución (modo offline o respaldo en CI).")

# Resultado de la extracción en formato vertical (fácil de leer)
display(fila.T.rename(columns={0: "valor"}))

if EN_COLAB and GUARDAR_CSV:  # descarga opcional solo dentro de Colab
    from google.colab import files
    files.download(ARCHIVO_CSV)

## 10. Conclusiones y trabajo futuro

**Elección del método**

| Método | Evaluación para este caso |
|---|---|
| **BeautifulSoup + requests** | Ligero y rápido, pero solo lee el HTML que entrega el servidor. La prueba de la sección 4 muestra si el título y los precios vienen en ese HTML; en este sitio el contenido del producto se construye en el navegador y puede haber redirecciones de verificación. |
| **Selenium** | **Elegido.** Ejecuta el JavaScript, espera a que aparezcan los datos y entrega el DOM final. Es más lento y pesado, pero es adecuado para una consulta puntual por mes. |
| **Scrapy** | Descartado para esta fase: está pensado para rastreos masivos y no renderiza JavaScript por sí solo. Sería la opción a evaluar si el proyecto escalara a cientos de artículos (junto con un middleware de renderizado). |

En la solución final **Selenium obtiene el HTML renderizado y BeautifulSoup lo analiza**, combinando lo mejor de ambos.

**Alcance y límites**

- Se extrae un único artículo, una vez por ejecución, con pausas y sin evadir ningún control del sitio.
- Los selectores CSS dependen del diseño actual de MercadoLibre; si el sitio lo cambia, hay que actualizarlos
  (la prueba de la sección 7 ayuda a distinguir un error de código de un cambio del sitio).
- Los precios pueden variar según la ubicación, el vendedor y las promociones vigentes.

**Trabajo futuro**

1. Programar una ejecución mensual y acumular el histórico en el CSV.
2. Graficar la serie temporal de `precio_con_descuento` y `descuento_pct_sitio`.
3. Ampliar a otros modelos de iPhone y comparar tendencias.